<a href="https://colab.research.google.com/github/saadchaudhary-dev/flyrank-intern-ml/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadchaudhary-dev/flyrank-intern-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Baseline Rule

The baseline assigns each webpage an Opportunity Score using only information that is available at the time of the decision. Pages are given a higher score if they:

have high search volume (greater potential audience),
have low impressions (currently underperforming),
have low CTR (users are not clicking often),
rank between positions 8 and 20, where a content refresh may improve visibility.

The pages are then ranked from highest to lowest opportunity score. This ranking is intended as decision support, not a guarantee that refreshing the page will improve performance.

Reason Codes
Code	Meaning
RC1	High search volume
RC2	Low impressions
RC3	Low CTR
RC4	Ranking position between 8–20
RC5	Multiple positive signals combined
Code Cell
print("Baseline Rule Loaded")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Code Cell
import pandas as pd
import os

df = pd.read_csv("starter.csv")      # Change filename if needed


# -----------------------------
# Normalization
# -----------------------------
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())


df["sv_norm"] = normalize(df["search_volume"])
df["imp_norm"] = normalize(df["impressions_90d"])
df["ctr_norm"] = normalize(df["ctr"])

# Position score
df["position_score"] = (
    (df["position"] >= 8) &
    (df["position"] <= 20)
).astype(int)

# -----------------------------
# Opportunity Score
# -----------------------------
df["opportunity_score"] = (
      0.40 * df["sv_norm"]
    + 0.30 * (1 - df["imp_norm"])
    + 0.20 * (1 - df["ctr_norm"])
    + 0.10 * df["position_score"]
)

# -----------------------------
# Reason Codes
# -----------------------------
def reason_codes(row):

    reasons = []

    if row["sv_norm"] > 0.70:
        reasons.append("RC1")

    if row["imp_norm"] < 0.30:
        reasons.append("RC2")

    if row["ctr_norm"] < 0.30:
        reasons.append("RC3")

    if 8 <= row["position"] <= 20:
        reasons.append("RC4")

    if len(reasons) >= 3:
        reasons.append("RC5")

    return ", ".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

# -----------------------------
# Rank Queue
# -----------------------------
baseline = df.sort_values(
    "opportunity_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(baseline.head(20))
print("\nCSV saved successfully.")

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Top-20 Review

The highest-ranked pages are recommended for content refresh because they satisfy several of the baseline conditions, such as high search demand, relatively low impressions, low click-through rate, or ranking near the first page of search results.

Confidence is moderate because the recommendations are based on observed historical metrics rather than future outcomes.

The recommendations could be wrong if:

seasonal traffic changes affected the metrics,
rankings changed after the data was collected,
the page already has a planned update,
external factors (competitors or Google updates) influenced performance.
Code Cell
top20 = baseline.head(20).copy()

top20["recommended_action"] = "Refresh Content"

top20["confidence"] = "Moderate"

top20["possible_failure"] = (
    "Seasonality, recent ranking changes, or external factors"
)

display(
    top20[
        [
            "url",
            "opportunity_score",
            "reason_codes",
            "recommended_action",
            "confidence",
            "possible_failure"
        ]
    ]
)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Weak Picks

Some pages in the ranked list may not actually benefit from a refresh. For example:

pages with temporary traffic drops,
seasonal pages,
pages affected by recent Google algorithm updates,
pages missing important data.

These recommendations should therefore be reviewed by a human before action is taken.

Leakage Check

The baseline uses only current observed SEO metrics.

It does not use:

future impressions,
future clicks,
manually assigned labels,
product outcome flags,
information collected after the scoring date.

Therefore, no intentional data leakage has been introduced into the baseline.

Code Cell
print("Leakage Check")

future_columns = [
    col for col in df.columns
    if "future" in col.lower()
]

print("Future columns found:", future_columns)

print("\nDataset Columns:")

for col in df.columns:
    print("-", col)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.